In [2]:
pip install gdown pandas confluent_kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 28.0 MB/s eta 0:00:00


In [3]:
from confluent_kafka.admin import AdminClient
from minio import Minio
from minio.deleteobjects import DeleteObject  # <--- Cập nhật thêm dòng này
import time

# --- CẤU HÌNH ---
KAFKA_HOST = '18.219.196.163:9092'
MINIO_ENDPOINT = 'play.min.io'
ACCESS_KEY = 'minioadmin'
SECRET_KEY = 'minioadmin'

# Danh sách TẤT CẢ Topic muốn xóa
TOPICS_TO_DELETE = [
    'weather_multi_city',
    'live_weather_api',
    'iot_final_data',
    'hourly_weather_data',
    'weather_history_data'
]

# Danh sách Bucket muốn xóa
BUCKETS_TO_DELETE = [
    'weather-data-demo',
    'weather-history-archive'
]

def wipe_minio():
    print(f"\n🗑️  [MINIO] Đang dọn dẹp dữ liệu...")
    try:
        client = Minio(MINIO_ENDPOINT, access_key=ACCESS_KEY, secret_key=SECRET_KEY, secure=True)

        for bucket_name in BUCKETS_TO_DELETE:
            if not client.bucket_exists(bucket_name):
                print(f"   ⚠️ Bucket '{bucket_name}' không tồn tại -> Bỏ qua.")
                continue

            # 1. Xóa tất cả file bên trong
            print(f"   🔥 Đang xóa file trong: {bucket_name}...")
            objects = client.list_objects(bucket_name, recursive=True)

            # SỬA LỖI Ở ĐÂY: Chuyển tên file thành đối tượng DeleteObject
            delete_list = [DeleteObject(x.object_name) for x in objects]

            if delete_list:
                errors = client.remove_objects(bucket_name, delete_list)
                error_count = 0
                for err in errors:
                    print(f"   ❌ Lỗi xóa file: {err}")
                    error_count += 1
                if error_count == 0:
                     print(f"   ✅ Đã xóa sạch file trong bucket.")

            # 2. Xóa vỏ Bucket
            try:
                client.remove_bucket(bucket_name)
                print(f"   ✅ Đã XÓA SỔ bucket: {bucket_name}")
            except Exception as e:
                print(f"   ❌ Không xóa được bucket {bucket_name} (Có thể còn file sót): {e}")

    except Exception as e:
        print(f"❌ Lỗi kết nối MinIO: {e}")

def wipe_kafka():
    print(f"\n🗑️  [KAFKA] Đang xóa các Topic...")
    try:
        admin_client = AdminClient({'bootstrap.servers': KAFKA_HOST})

        # Gửi lệnh xóa
        fs = admin_client.delete_topics(TOPICS_TO_DELETE, operation_timeout=30)

        # Chờ kết quả
        for topic, f in fs.items():
            try:
                f.result() # Chờ xác nhận
                print(f"   ✅ Đã XÓA SỔ topic: {topic}")
            except Exception as e:
                # Lỗi này thường do topic không tồn tại, không sao cả
                print(f"   ⚠️ Không tìm thấy topic '{topic}' để xóa.")

        print("   ⏳ Đang chờ Kafka dọn rác (5s)...")
        time.sleep(5)
    except Exception as e:
        print(f"❌ Lỗi kết nối Kafka: {e}")

def main():
    print("🛑 CẢNH BÁO: Code này sẽ xóa vĩnh viễn dữ liệu cũ (Phiên bản Fix lỗi).")
    confirm = input("👉 Gõ 'DELETE' để xác nhận xóa: ")

    if confirm == 'DELETE':
        wipe_minio()
        wipe_kafka()
        print("\n✨ HỆ THỐNG ĐÃ SẠCH SẼ HOÀN TOÀN ✨")
    else:
        print("❌ Mã xác nhận sai. Đã hủy.")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'minio'

In [25]:
from confluent_kafka import Consumer, TopicPartition
from confluent_kafka.admin import AdminClient
from minio import Minio

# --- 1. CẤU HÌNH ---
KAFKA_HOST = '18.219.196.163:9092'
MINIO_ENDPOINT = 'play.min.io'
ACCESS_KEY = 'minioadmin'
SECRET_KEY = 'minioadmin'

# Các Topic/Bucket chính của bạn (để in đậm cho dễ nhìn)
MY_TOPICS = ['weather_multi_city', 'weather_history_data']
MY_BUCKETS = ['weather-data-demo', 'weather-history-archive']

def check_kafka():
    print(f"\n{'='*20} 🔍 KIỂM TRA KAFKA {'='*20}")
    print(f"📡 Kết nối: {KAFKA_HOST}")

    try:
        # 1. Lấy danh sách Topic
        admin_client = AdminClient({'bootstrap.servers': KAFKA_HOST})
        metadata = admin_client.list_topics(timeout=10)

        # Tạo consumer tạm để đếm số tin nhắn
        conf = {'bootstrap.servers': KAFKA_HOST, 'group.id': 'checker-tool', 'auto.offset.reset': 'latest'}
        consumer = Consumer(conf)

        topics = list(metadata.topics.keys())
        print(f"👉 Tìm thấy tổng {len(topics)} Topic.")

        print(f"\n{'TOPIC NAME':<30} | {'PARTITIONS':<10} | {'MESSAGES (Ước tính)':<15}")
        print("-" * 65)

        for topic in topics:
            # Bỏ qua các topic hệ thống của Kafka (thường bắt đầu bằng _)
            if topic.startswith('_'): continue

            # Lấy thông tin Partition
            partitions = metadata.topics[topic].partitions
            part_count = len(partitions)

            # Đếm số tin nhắn (Lấy Offset cao nhất - Offset thấp nhất)
            total_msgs = 0
            for p_id in partitions:
                try:
                    low, high = consumer.get_watermark_offsets(TopicPartition(topic, p_id), timeout=2.0)
                    total_msgs += (high - low)
                except:
                    pass

            # Đánh dấu sao cho topic của mình
            marker = "⭐ " if topic in MY_TOPICS else "   "
            print(f"{marker}{topic:<27} | {part_count:<10} | {total_msgs:<15}")

        consumer.close()

    except Exception as e:
        print(f"❌ Không thể kết nối Kafka: {e}")

def check_minio():
    print(f"\n{'='*20} 🔍 KIỂM TRA MINIO {'='*20}")
    print(f"📡 Kết nối: {MINIO_ENDPOINT}")

    try:
        client = Minio(MINIO_ENDPOINT, access_key=ACCESS_KEY, secret_key=SECRET_KEY, secure=True)

        # 1. Lấy danh sách Bucket
        buckets = client.list_buckets()
        print(f"👉 Tìm thấy tổng {len(buckets)} Bucket.\n")

        for bucket in buckets:
            b_name = bucket.name

            # Chỉ soi kỹ bucket của bạn, bucket người khác thì bỏ qua cho đỡ rối
            if b_name not in MY_BUCKETS:
                # print(f"   📂 {b_name} (Bỏ qua)")
                continue

            print(f"📂 BUCKET: {b_name}")

            # 2. Đếm file trong bucket
            objects = list(client.list_objects(b_name, recursive=True))
            count = len(objects)

            if count == 0:
                print("   └── (Trống rỗng)")
            else:
                print(f"   └── Chứa: {count} file")
                # In thử 3 file đầu tiên
                print("       Ví dụ:")
                for obj in objects[:3]:
                    size_kb = obj.size / 1024
                    print(f"       - {obj.object_name} ({size_kb:.1f} KB)")
                if count > 3:
                    print(f"       ... và {count - 3} file khác.")
            print("-" * 40)

    except Exception as e:
        print(f"❌ Không thể kết nối MinIO: {e}")

if __name__ == "__main__":
    check_kafka()
    check_minio()
    print("\n✅ HOÀN TẤT KIỂM TRA.")


==================== 🔍 KIỂM TRA KAFKA ====================
📡 Kết nối: 18.219.196.163:9092
👉 Tìm thấy tổng 3 Topic.

TOPIC NAME                     | PARTITIONS | MESSAGES (Ước tính)
-----------------------------------------------------------------
⭐ weather_history_data        | 1          | 914592         
   weather_forecast_data       | 1          | 1344           

==================== 🔍 KIỂM TRA MINIO ====================
📡 Kết nối: play.min.io
👉 Tìm thấy tổng 221 Bucket.

📂 BUCKET: weather-history-archive
   └── Chứa: 19 file
       Ví dụ:
       - history_data_part_001.csv (7012.3 KB)
       - history_data_part_002.csv (7012.2 KB)
       - history_data_part_003.csv (7009.3 KB)
       ... và 16 file khác.
----------------------------------------

✅ HOÀN TẤT KIỂM TRA.


In [4]:
import gdown
import pandas as pd
import json
import time
import os
from confluent_kafka import Producer

# --- 1. CẤU HÌNH ---
# Link Drive của bạn
DRIVE_URL = 'https://drive.google.com/file/d/1FPIMercG7BVwM2Qj_H8LRuVZQ5epJ78Y/view?usp=drive_link'
# ID file (Lấy từ link trên)
FILE_ID = '1FPIMercG7BVwM2Qj_H8LRuVZQ5epJ78Y'
OUTPUT_FILE = 'Vietnam_IoT_Final_Dataset.csv'

# Cấu hình Kafka
KAFKA_HOST = '18.219.196.163:9092'  # IP Public EC2 của bạn
TOPIC_NAME = 'weather_history_data' # Topic riêng cho dữ liệu lịch sử

def download_data():
    """Hàm tải file từ Google Drive"""
    print(f"📥 Đang tải dữ liệu từ Google Drive...")

    # URL tải trực tiếp của Gdown
    url = f'https://drive.google.com/uc?id={FILE_ID}'

    # Tải file (tự động xử lý xác thực Google)
    if not os.path.exists(OUTPUT_FILE):
        gdown.download(url, OUTPUT_FILE, quiet=False)
        print(f"✅ Tải xong file: {OUTPUT_FILE}")
    else:
        print(f"✅ File {OUTPUT_FILE} đã tồn tại, bỏ qua bước tải.")

def delivery_report(err, msg):
    """Callback báo lỗi nếu gửi thất bại"""
    if err is not None:
        print(f'❌ Gửi lỗi: {err}')

def main():
    # 1. Tải file về máy
    download_data()

    # 2. Đọc file bằng Pandas
    print(f"📖 Đang đọc file CSV và chuyển đổi dữ liệu...")
    try:
        # Đọc file CSV
        df = pd.read_csv(OUTPUT_FILE)

        # Xử lý dữ liệu: Chuyển các cột object/datetime thành string để JSON không lỗi
        # (Tùy dữ liệu thực tế, code này ép kiểu chung cho an toàn)
        df = df.astype(object).where(pd.notnull(df), None)

        records = df.to_dict(orient='records')
        total_records = len(records)
        print(f"✅ Đã chuẩn bị {total_records} dòng dữ liệu.")

    except Exception as e:
        print(f"❌ Lỗi khi đọc file CSV: {e}")
        return

    # 3. Khởi tạo Kafka Producer
    conf = {
        'bootstrap.servers': KAFKA_HOST,
        'client.id': 'drive-history-producer',
        'queue.buffering.max.messages': 500000, # Tăng bộ đệm
        'queue.buffering.max.ms': 1000,
        'batch.num.messages': 10000
    }
    producer = Producer(conf)

    print(f"🚀 Bắt đầu bắn {total_records} bản tin vào Kafka Topic: '{TOPIC_NAME}'")
    print("-" * 50)

    start_time = time.time()

    # 4. Gửi dữ liệu vòng lặp
    for i, row in enumerate(records):
        try:
            # Chuyển row thành JSON string
            val = json.dumps(row, default=str).encode('utf-8')

            # Gửi bất đồng bộ (Async)
            producer.produce(TOPIC_NAME, value=val, callback=delivery_report)

            # Flush định kỳ để tránh tràn RAM cục bộ
            if (i + 1) % 10000 == 0:
                producer.poll(0)
                print(f"   -> Đã gửi: {i + 1}/{total_records} dòng...")

        except BufferError:
            # Nếu bộ đệm đầy, chờ 1 chút rồi gửi tiếp
            print("⚠️ Bộ đệm đầy, đang chờ xả...")
            producer.poll(1)
            producer.produce(TOPIC_NAME, value=json.dumps(row, default=str).encode('utf-8'), callback=delivery_report)

    # Gửi nốt những tin còn lại trong hàng đợi
    print("⏳ Đang đẩy nốt dữ liệu còn lại...")
    producer.flush()

    duration = time.time() - start_time
    print("-" * 50)
    print(f"🎉 HOÀN TẤT! Đã gửi {total_records} dòng trong {duration:.2f} giây.")

if __name__ == "__main__":
    main()

📥 [1/3] Đang tải dữ liệu gốc...


Downloading...
From (original): https://drive.google.com/uc?id=1FPIMercG7BVwM2Qj_H8LRuVZQ5epJ78Y
From (redirected): https://drive.google.com/uc?id=1FPIMercG7BVwM2Qj_H8LRuVZQ5epJ78Y&confirm=t&uuid=67ea697d-15d6-4e64-9ac5-4e348958bd99
To: /content/temp_original_data.csv
100%|██████████| 164M/164M [00:02<00:00, 77.4MB/s]



🔄 [2/3] Đang lấy dữ liệu ngày 2026-01-31...
   ✅ Đã lấy: Ha Noi
   ✅ Đã lấy: Hai Phong
   ✅ Đã lấy: Da Nang
   ✅ Đã lấy: Can Tho


KeyError: "['apparent_temp'] not in index"

In [5]:
import gdown
import pandas as pd
import requests
import os
from datetime import datetime

# ==========================================
# CẤU HÌNH
# ==========================================
DRIVE_FILE_ID = '1FPIMercG7BVwM2Qj_H8LRuVZQ5epJ78Y'
TEMP_FILE     = 'temp_original_data.csv'
FINAL_FILE    = 'Final_Weather_Full_Updated.csv'

# Ngày cần lấy thêm
MISSING_DATE = "2026-01-31"
ARCHIVE_API_URL = "https://archive-api.open-meteo.com/v1/archive"

LOCATIONS = [
    {"city": "Ha Noi",    "region": "North",   "sensor_id": "SENSOR_HA_NOI_01",   "lat": 21.0285, "lon": 105.8542},
    {"city": "Hai Phong", "region": "North",   "sensor_id": "SENSOR_HAI_PHONG_01","lat": 20.8449, "lon": 106.6881},
    {"city": "Da Nang",   "region": "Central", "sensor_id": "SENSOR_DA_NANG_01",  "lat": 16.0471, "lon": 108.2068},
    {"city": "Can Tho",   "region": "South",   "sensor_id": "SENSOR_CAN_THO_01",  "lat": 10.0452, "lon": 105.7469}
]

API_VARIABLES = [
    "temperature_2m", "relative_humidity_2m", "apparent_temperature",
    "weather_code", "pressure_msl", "rain", "wind_speed_10m",
    "wind_direction_10m", "wind_gusts_10m", "shortwave_radiation",
    "visibility", "cloud_cover", "is_day"
]

# --- SỬA LẠI MAPPING CHUẨN THEO FILE CỦA BẠN ---
COLUMN_MAPPING = {
    "time": "timestamp",
    "temperature_2m": "temperature",
    "relative_humidity_2m": "humidity",
    "apparent_temperature": "apparent_temp", # <-- ĐÃ SỬA: apparent_temp
    "weather_code": "weather_code",
    "pressure_msl": "pressure",
    "rain": "rain",
    "wind_speed_10m": "wind_speed",
    "wind_direction_10m": "wind_direction",
    "wind_gusts_10m": "wind_gusts",
    "shortwave_radiation": "solar_radiation",
    "visibility": "visibility",
    "cloud_cover": "cloud_cover",
    "is_day": "is_day"
}

# ==========================================
# XỬ LÝ
# ==========================================
def main():
    # 1. Tải file gốc
    print(f"📥 [1/3] Đang tải dữ liệu gốc...")
    if not os.path.exists(TEMP_FILE):
        url = f'https://drive.google.com/uc?id={DRIVE_FILE_ID}'
        gdown.download(url, TEMP_FILE, quiet=False)

    try:
        df_main = pd.read_csv(TEMP_FILE)
        # Loại bỏ cột rác (Unnamed) nếu có
        df_main = df_main.loc[:, ~df_main.columns.str.contains('^Unnamed')]
    except Exception as e:
        print(f"❌ Lỗi đọc file: {e}")
        return

    # 2. Tải dữ liệu mới
    print(f"\n🔄 [2/3] Đang lấy dữ liệu ngày {MISSING_DATE}...")

    # Kiểm tra xem ngày này đã có chưa
    if not df_main['timestamp'].astype(str).str.contains(MISSING_DATE).any():
        new_chunks = []
        for loc in LOCATIONS:
            params = {
                "latitude": loc["lat"], "longitude": loc["lon"],
                "start_date": MISSING_DATE, "end_date": MISSING_DATE,
                "hourly": ",".join(API_VARIABLES), "timezone": "Asia/Bangkok"
            }
            try:
                r = requests.get(ARCHIVE_API_URL, params=params, timeout=10)
                if r.status_code == 200:
                    data = r.json()
                    df_temp = pd.DataFrame(data.get("hourly", {}))

                    # --- BƯỚC QUAN TRỌNG: ĐỔI TÊN CỘT ---
                    df_temp.rename(columns=COLUMN_MAPPING, inplace=True)

                    # Format Timestamp
                    df_temp["timestamp"] = pd.to_datetime(df_temp["timestamp"]).dt.strftime('%Y-%m-%d %H:%M:%S') + "+07:00"

                    # Thêm Metadata
                    df_temp["city"] = loc["city"]
                    df_temp["region"] = loc["region"]
                    df_temp["sensor_id"] = loc["sensor_id"]

                    new_chunks.append(df_temp)
                    print(f"   ✅ Đã lấy: {loc['city']}")
            except:
                print(f"   ❌ Lỗi lấy: {loc['city']}")

        if new_chunks:
            df_new = pd.concat(new_chunks, ignore_index=True)

            # --- BƯỚC KHẮC PHỤC KEYERROR ---
            # Chỉ lấy đúng các cột đang có trong file gốc để đảm bảo khớp 100%
            # Dùng .intersection để lấy giao của 2 tập cột
            common_cols = [col for col in df_main.columns if col in df_new.columns]

            # Nếu thiếu cột quan trọng thì báo lỗi, nếu không thì tiếp tục
            df_new = df_new[common_cols]

            # Gộp
            df_final = pd.concat([df_main, df_new], ignore_index=True)
            print(f"   -> Đã thêm thành công.")
        else:
            df_final = df_main
    else:
        print("   ⚠️ Dữ liệu ngày này đã có sẵn.")
        df_final = df_main

    # 3. Lưu file
    print(f"\n💾 [3/3] Xuất file CSV: {FINAL_FILE}")
    df_final.to_csv(FINAL_FILE, index=False)

    print("-" * 40)
    print(f"🎉 THÀNH CÔNG! File đã được lưu.")
    print(f"📂 File: {FINAL_FILE}")
    print(f"📊 Tổng số dòng: {len(df_final)}")
    print("-" * 40)

    try:
        from google.colab import files
        files.download(FINAL_FILE)
    except:
        pass

if __name__ == "__main__":
    main()

📥 [1/3] Đang tải dữ liệu gốc...

🔄 [2/3] Đang lấy dữ liệu ngày 2026-01-31...
   ✅ Đã lấy: Ha Noi
   ✅ Đã lấy: Hai Phong
   ✅ Đã lấy: Da Nang
   ✅ Đã lấy: Can Tho
   -> Đã thêm thành công.

💾 [3/3] Xuất file CSV: Final_Weather_Full_Updated.csv
----------------------------------------
🎉 THÀNH CÔNG! File đã được lưu.
📂 File: Final_Weather_Full_Updated.csv
📊 Tổng số dòng: 914688
----------------------------------------


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
from confluent_kafka import Consumer
from minio import Minio
import json
import csv
import io
import time
import sys

# --- 1. CẤU HÌNH ---
# KAFKA
KAFKA_HOST = '18.219.196.163:9092'  # IP EC2
TOPIC_NAME = 'weather_history_data' # Topic chứa dữ liệu lịch sử
GROUP_ID = 'history-saver-group-final' # Tên nhóm (đổi mới để đọc từ đầu)

# MINIO
MINIO_ENDPOINT = 'play.min.io'
ACCESS_KEY = 'minioadmin'
SECRET_KEY = 'minioadmin'
BUCKET_NAME = 'weather-history-archive' # Bucket lưu trữ

# CẤU HÌNH GÓI (QUAN TRỌNG)
BATCH_SIZE = 50000  # Cứ đủ 50.000 dòng thì lưu thành 1 file
IDLE_TIMEOUT = 10   # Nếu chờ 10 giây mà không có tin mới -> Lưu luôn phần còn lại

def save_batch_to_minio(minio_client, buffer_data, file_index):
    if not buffer_data:
        return

    # Tạo tên file: history_part_001.csv
    file_name = f"history_data_part_{file_index:03d}.csv"
    print(f"   💾 Đang đóng gói {len(buffer_data)} dòng vào file: {file_name}...")

    try:
        # Lấy header động từ dòng đầu tiên
        headers = buffer_data[0].keys()

        # Tạo file CSV trong bộ nhớ (không cần lưu xuống ổ cứng máy)
        csv_buffer = io.StringIO()
        writer = csv.DictWriter(csv_buffer, fieldnames=headers)
        writer.writeheader()
        writer.writerows(buffer_data)

        # Upload lên MinIO
        csv_bytes = csv_buffer.getvalue().encode('utf-8')
        minio_client.put_object(
            BUCKET_NAME,
            file_name,
            io.BytesIO(csv_bytes),
            length=len(csv_bytes),
            content_type='application/csv'
        )
        print(f"   ✅ ĐÃ LƯU XONG: {file_name} lên MinIO")
    except Exception as e:
        print(f"   ❌ Lỗi lưu file {file_name}: {e}")

def main():
    print(f"🔥 Bắt đầu tiến trình Consumer Lịch sử...")

    # 1. Kết nối MinIO
    client = Minio(MINIO_ENDPOINT, access_key=ACCESS_KEY, secret_key=SECRET_KEY, secure=True)
    if not client.bucket_exists(BUCKET_NAME):
        client.make_bucket(BUCKET_NAME)
        print(f"   + Đã tạo bucket mới: {BUCKET_NAME}")

    # 2. Kết nối Kafka
    conf = {
        'bootstrap.servers': KAFKA_HOST,
        'group.id': GROUP_ID,
        'auto.offset.reset': 'earliest', # Bắt buộc: Đọc từ tin nhắn đầu tiên trong quá khứ
        'enable.auto.commit': True
    }
    consumer = Consumer(conf)
    consumer.subscribe([TOPIC_NAME])

    print(f"🎧 Đang lắng nghe Topic '{TOPIC_NAME}'...")
    print(f"📦 Chế độ: Batch {BATCH_SIZE} dòng/file.")
    print("-" * 50)

    buffer = []
    file_count = 1
    last_msg_time = time.time()

    try:
        while True:
            # Poll dữ liệu (chờ tối đa 1 giây)
            msg = consumer.poll(1.0)

            # --- LOGIC XỬ LÝ KHI KHÔNG CÓ TIN NHẮN MỚI ---
            if msg is None:
                current_time = time.time()
                # Nếu buffer có dữ liệu VÀ đã chờ quá lâu (IDLE_TIMEOUT) -> Lưu nốt
                if len(buffer) > 0 and (current_time - last_msg_time > IDLE_TIMEOUT):
                    print(f"⚠️ Đã hết dữ liệu mới trong {IDLE_TIMEOUT}s. Lưu phần còn lại...")
                    save_batch_to_minio(client, buffer, file_count)
                    buffer = []
                    file_count += 1
                continue

            if msg.error():
                print(f"Lỗi Kafka: {msg.error()}")
                continue

            # --- LOGIC XỬ LÝ TIN NHẮN ---
            try:
                # Cập nhật thời gian lần cuối nhận tin
                last_msg_time = time.time()

                # Giải mã
                data = json.loads(msg.value().decode('utf-8'))
                buffer.append(data)

                # Nếu buffer đầy -> Lưu file
                if len(buffer) >= BATCH_SIZE:
                    save_batch_to_minio(client, buffer, file_count)
                    buffer = [] # Reset buffer
                    file_count += 1

            except Exception as e:
                print(f"Lỗi giải mã JSON: {e}")

    except KeyboardInterrupt:
        print("\n🛑 Người dùng dừng chương trình.")
        # Lưu nốt dữ liệu còn sót lại nếu có
        if len(buffer) > 0:
            print("💾 Đang lưu nốt dữ liệu cuối cùng trước khi tắt...")
            save_batch_to_minio(client, buffer, file_count)
    finally:
        consumer.close()

if __name__ == "__main__":
    main()

🔥 Bắt đầu tiến trình Consumer Lịch sử...
   + Đã tạo bucket mới: weather-history-archive
🎧 Đang lắng nghe Topic 'weather_history_data'...
📦 Chế độ: Batch 50000 dòng/file.
--------------------------------------------------
   💾 Đang đóng gói 50000 dòng vào file: history_data_part_001.csv...
   ✅ ĐÃ LƯU XONG: history_data_part_001.csv lên MinIO
   💾 Đang đóng gói 50000 dòng vào file: history_data_part_002.csv...
   ✅ ĐÃ LƯU XONG: history_data_part_002.csv lên MinIO
   💾 Đang đóng gói 50000 dòng vào file: history_data_part_003.csv...
   ✅ ĐÃ LƯU XONG: history_data_part_003.csv lên MinIO
   💾 Đang đóng gói 50000 dòng vào file: history_data_part_004.csv...
   ✅ ĐÃ LƯU XONG: history_data_part_004.csv lên MinIO
   💾 Đang đóng gói 50000 dòng vào file: history_data_part_005.csv...
   ✅ ĐÃ LƯU XONG: history_data_part_005.csv lên MinIO
   💾 Đang đóng gói 50000 dòng vào file: history_data_part_006.csv...
   ✅ ĐÃ LƯU XONG: history_data_part_006.csv lên MinIO
   💾 Đang đóng gói 50000 dòng vào file: 

In [20]:
from confluent_kafka.admin import AdminClient, NewTopic
import time

# --- CẤU HÌNH ---
KAFKA_HOST = '18.219.196.163:9092'
TOPIC_NAME = 'weather_forecast_data'

def create_topic():
    print(f"🔧 Đang kết nối đến Kafka: {KAFKA_HOST}...")
    admin_client = AdminClient({'bootstrap.servers': KAFKA_HOST})

    # Tạo Topic mới: 1 Partition, 1 Replication (cấu hình cơ bản)
    topic_list = [NewTopic(TOPIC_NAME, num_partitions=1, replication_factor=1)]

    # Gửi lệnh tạo
    fs = admin_client.create_topics(topic_list)

    # Chờ kết quả
    for topic, f in fs.items():
        try:
            f.result()  # Kết quả trả về None nếu thành công
            print(f"✅ Đã tạo thành công Topic: '{topic}'")
        except Exception as e:
            # Nếu lỗi là "TopicExistsError" thì không sao, nghĩa là có rồi
            if "TopicExists" in str(e):
                print(f"⚠️ Topic '{topic}' đã tồn tại từ trước (Không cần tạo lại).")
            else:
                print(f"❌ Lỗi tạo topic: {e}")

    print("-" * 40)
    print("⏳ Đang đợi Kafka đồng bộ (3s)...")
    time.sleep(3)
    print("🚀 SẴN SÀNG! Bạn có thể chạy Consumer ngay bây giờ.")

if __name__ == "__main__":
    create_topic()

🔧 Đang kết nối đến Kafka: 18.219.196.163:9092...
❌ Lỗi tạo topic: KafkaError{code=TOPIC_ALREADY_EXISTS,val=36,str="Topic 'weather_forecast_data' already exists."}
----------------------------------------
⏳ Đang đợi Kafka đồng bộ (3s)...
🚀 SẴN SÀNG! Bạn có thể chạy Consumer ngay bây giờ.


In [21]:
import gdown
import pandas as pd
import json
import time
import os
from confluent_kafka import Producer

# --- CẤU HÌNH ---
KAFKA_HOST = '18.219.196.163:9092'
TOPIC_NAME = 'weather_forecast_data'

# Danh sách 4 file của bạn (ID lấy từ link bạn gửi)
FILES_INFO = [
    {"city": "Can Tho",   "id": "1ZWEG8J4VnWclhkglDSTpCDbKXA_wfkpE", "file": "forecast_cantho.csv"},
    {"city": "Da Nang",   "id": "1m9HNPRyVbeTO0HcTnvPnBbGaiGhtiLOt", "file": "forecast_danang.csv"},
    {"city": "Ha Noi",    "id": "1T4OFFsBX34Gtqgq_nXA7TUYeW362Y1C8", "file": "forecast_hanoi.csv"},
    {"city": "Hai Phong", "id": "1jB1OVn_K6f4HaY1hiJMGEPcq4-DTYUHK", "file": "forecast_haiphong.csv"}
]

def delivery_report(err, msg):
    if err is not None:
        print(f'❌ Gửi lỗi: {err}')

def main():
    # 1. Khởi tạo Producer
    producer = Producer({'bootstrap.servers': KAFKA_HOST})

    print(f"🚀 BẮT ĐẦU XỬ LÝ 4 FILE DỰ BÁO...")
    print("-" * 50)

    for item in FILES_INFO:
        city = item['city']
        file_id = item['id']
        file_name = item['file']

        # A. Tải file từ Drive
        print(f"\n📥 Đang tải dữ liệu cho: {city}...")
        url = f'https://drive.google.com/uc?id={file_id}'
        if not os.path.exists(file_name):
            gdown.download(url, file_name, quiet=False)

        # B. Đọc file
        try:
            df = pd.read_csv(file_name)

            # Thêm cột 'city' nếu file chưa có (để phân biệt khi gộp chung)
            if 'city' not in df.columns:
                df['city'] = city

            records = df.to_dict(orient='records')
            count = len(records)
            print(f"   📖 Đã đọc {count} dòng dự báo của {city}.")

            # C. Gửi Kafka
            print(f"   📡 Đang gửi vào Kafka...")
            for row in records:
                # Chuyển row thành JSON
                val = json.dumps(row, default=str).encode('utf-8')
                producer.produce(TOPIC_NAME, value=val, callback=delivery_report)

            producer.flush()
            print(f"   ✅ Hoàn tất gửi {city}!")

        except Exception as e:
            print(f"❌ Lỗi xử lý file {city}: {e}")

    print("-" * 50)
    print("🎉 TẤT CẢ HOÀN TẤT! Đã đẩy xong 4 thành phố.")

if __name__ == "__main__":
    main()

🚀 BẮT ĐẦU XỬ LÝ 4 FILE DỰ BÁO...
--------------------------------------------------

📥 Đang tải dữ liệu cho: Can Tho...
   📖 Đã đọc 168 dòng dự báo của Can Tho.
   📡 Đang gửi vào Kafka...
   ✅ Hoàn tất gửi Can Tho!

📥 Đang tải dữ liệu cho: Da Nang...
   📖 Đã đọc 168 dòng dự báo của Da Nang.
   📡 Đang gửi vào Kafka...
   ✅ Hoàn tất gửi Da Nang!

📥 Đang tải dữ liệu cho: Ha Noi...
   📖 Đã đọc 168 dòng dự báo của Ha Noi.
   📡 Đang gửi vào Kafka...
   ✅ Hoàn tất gửi Ha Noi!

📥 Đang tải dữ liệu cho: Hai Phong...
   📖 Đã đọc 168 dòng dự báo của Hai Phong.
   📡 Đang gửi vào Kafka...
   ✅ Hoàn tất gửi Hai Phong!
--------------------------------------------------
🎉 TẤT CẢ HOÀN TẤT! Đã đẩy xong 4 thành phố.


In [24]:
import gdown
from minio import Minio
import os

# --- 1. CẤU HÌNH KẾT NỐI EC2 ---
MINIO_ENDPOINT = '18.219.196.163:9000' # IP + Cổng 9000
ACCESS_KEY = 'minioadmin'             # User
SECRET_KEY = 'minioadmin'             # Pass
USE_SSL = False                       # Dùng IP nên tắt SSL
BUCKET_NAME = 'weather-forecast-raw'  # Bucket chứa file thô (Source)

# --- 2. DANH SÁCH 4 FILE CSV ---
CSV_FILES = [
    {"city": "Can Tho",   "id": "1ZWEG8J4VnWclhkglDSTpCDbKXA_wfkpE", "file": "forecast_cantho.csv"},
    {"city": "Da Nang",   "id": "1m9HNPRyVbeTO0HcTnvPnBbGaiGhtiLOt", "file": "forecast_danang.csv"},
    {"city": "Ha Noi",    "id": "1T4OFFsBX34Gtqgq_nXA7TUYeW362Y1C8", "file": "forecast_hanoi.csv"},
    {"city": "Hai Phong", "id": "1jB1OVn_K6f4HaY1hiJMGEPcq4-DTYUHK", "file": "forecast_haiphong.csv"}
]

def main():
    print(f"🔌 Đang kết nối MinIO tại {MINIO_ENDPOINT}...")
    try:
        client = Minio(MINIO_ENDPOINT, access_key=ACCESS_KEY, secret_key=SECRET_KEY, secure=USE_SSL)

        # Tạo bucket riêng cho file raw
        if not client.bucket_exists(BUCKET_NAME):
            client.make_bucket(BUCKET_NAME)
            print(f"📁 Đã tạo Bucket mới: {BUCKET_NAME}")
        else:
            print(f"📁 Bucket '{BUCKET_NAME}' đã có sẵn.")

        print("-" * 50)
        print("🚀 BẮT ĐẦU TẢI VÀ UPLOAD CSV...")

        for item in CSV_FILES:
            file_name = item['file']
            file_id = item['id']
            city = item['city']

            # 1. Tải từ Drive (nếu chưa có)
            if not os.path.exists(file_name):
                print(f"\n📥 Đang tải file của {city}...")
                url = f'https://drive.google.com/uc?id={file_id}'
                gdown.download(url, file_name, quiet=False)

            # 2. Upload thẳng lên MinIO
            print(f"   ☁️  Đang đẩy file '{file_name}' lên MinIO...")
            client.fput_object(
                BUCKET_NAME,
                file_name,      # Tên file trên MinIO
                file_name,      # Đường dẫn file ở máy
                content_type='application/csv'
            )
            print(f"   ✅ OK!")

        print("\n" + "=" * 50)
        print(f"🎉 HOÀN TẤT! 4 file CSV đã nằm trong bucket '{BUCKET_NAME}'.")
        print("👉 Bạn có thể kiểm tra tại: http://18.219.196.163:9001")

    except Exception as e:
        print(f"❌ Lỗi: {e}")

if __name__ == "__main__":
    main()

🔌 Đang kết nối MinIO tại 18.219.196.163:9000...
📁 Đã tạo Bucket mới: weather-forecast-raw
--------------------------------------------------
🚀 BẮT ĐẦU TẢI VÀ UPLOAD CSV...
   ☁️  Đang đẩy file 'forecast_cantho.csv' lên MinIO...
   ✅ OK!
   ☁️  Đang đẩy file 'forecast_danang.csv' lên MinIO...
   ✅ OK!
   ☁️  Đang đẩy file 'forecast_hanoi.csv' lên MinIO...
   ✅ OK!
   ☁️  Đang đẩy file 'forecast_haiphong.csv' lên MinIO...
   ✅ OK!

🎉 HOÀN TẤT! 4 file CSV đã nằm trong bucket 'weather-forecast-raw'.
👉 Bạn có thể kiểm tra tại: http://18.219.196.163:9001


In [23]:
import gdown
from minio import Minio
import os

# --- CẤU HÌNH CHUẨN CHO EC2 CỦA BẠN ---
MINIO_ENDPOINT = '18.219.196.163:9000' # <--- IP của bạn + cổng 9000
ACCESS_KEY = 'minioadmin'             # <--- Mật khẩu MinIO của bạn
SECRET_KEY = 'minioadmin'             # <--- Mật khẩu MinIO của bạn
BUCKET_NAME = 'weather-models'
USE_SSL = False                       # <--- Bắt buộc False khi dùng IP

# Danh sách file
MODELS_INFO = [
    {"name": "best_model_Can_Tho.pkl",   "id": "11DVAZDpEMYyKQUGZykqFIBS5bwWVGtwa"},
    {"name": "best_model_Da_Nang.pkl",   "id": "17xMz3XWcCA-rjVcyRwn4THcW6TAxfOjD"},
    {"name": "best_model_Ha_Noi.pkl",    "id": "1_egOcy0IMWlKCV71k5Uw3p9eeoaantZj"},
    {"name": "best_model_Hai_Phong.pkl", "id": "1qxSGftXvSeUPU3Sgd593cwe8wauQz64_"}
]

def main():
    print(f"🔌 Đang kết nối MinIO tại EC2: {MINIO_ENDPOINT}...")
    try:
        # Lưu ý: secure=USE_SSL (là False)
        client = Minio(MINIO_ENDPOINT, access_key=ACCESS_KEY, secret_key=SECRET_KEY, secure=USE_SSL)

        # Kiểm tra kết nối bằng cách tạo bucket
        if not client.bucket_exists(BUCKET_NAME):
            client.make_bucket(BUCKET_NAME)
            print(f"📁 Đã tạo Bucket mới trên EC2: {BUCKET_NAME}")
        else:
            print(f"📁 Bucket '{BUCKET_NAME}' đã có sẵn trên EC2.")

        print("-" * 50)

        for item in MODELS_INFO:
            file_name = item['name']

            # (Phần tải file giữ nguyên, nếu máy có rồi thì nó tự bỏ qua)
            if not os.path.exists(file_name):
                url = f'https://drive.google.com/uc?id={item["id"]}'
                gdown.download(url, file_name, quiet=False)

            # Upload
            print(f"   ☁️  Đang đẩy {file_name} lên EC2...")
            client.fput_object(BUCKET_NAME, file_name, file_name, content_type='application/octet-stream')
            print(f"   ✅ OK!")

        print("\n🎉 HOÀN TẤT! Hãy vào http://18.219.196.163:9001 để kiểm tra.")

    except Exception as e:
        print(f"❌ Lỗi kết nối: {e}")
        print("💡 Gợi ý: Kiểm tra lại Access Key / Secret Key xem có đúng không?")

if __name__ == "__main__":
    main()

🔌 Đang kết nối MinIO tại EC2: 18.219.196.163:9000...
📁 Đã tạo Bucket mới trên EC2: weather-models
--------------------------------------------------
   ☁️  Đang đẩy best_model_Can_Tho.pkl lên EC2...
   ✅ OK!
   ☁️  Đang đẩy best_model_Da_Nang.pkl lên EC2...
   ✅ OK!
   ☁️  Đang đẩy best_model_Ha_Noi.pkl lên EC2...
   ✅ OK!
   ☁️  Đang đẩy best_model_Hai_Phong.pkl lên EC2...
   ✅ OK!

🎉 HOÀN TẤT! Hãy vào http://18.219.196.163:9001 để kiểm tra.


In [26]:
import pandas as pd
import numpy as np
import io
import joblib
import json
from minio import Minio
from lightgbm import LGBMRegressor
from datetime import datetime, timedelta

# --- 1. CẤU HÌNH ---
MINIO_CONF = {
    "endpoint": "18.219.196.163:9000",
    "access_key": "minioadmin",
    "secret_key": "minioadmin",
    "secure": False
}

# Bucket
BUCKET_HISTORY = "weather-data-history" # Chứa file lịch sử lớn (nếu có)
BUCKET_NEW = "weather-forecast-raw"     # Chứa dữ liệu API mới về (3h/lần)
BUCKET_MODEL = "weather-models"         # Nơi lưu model .pkl
BUCKET_OUTPUT = "weather-forecast-results" # Nơi lưu kết quả dự báo

# Cấu hình thành phố
CITIES = ["Can Tho", "Da Nang", "Ha Noi", "Hai Phong"]

# Các Features quan trọng (PHẢI GIỐNG HỆT LÚC BẠN THIẾT KẾ MODEL)
# Dựa trên code cũ của bạn:
LAG_PERIODS = [1, 2, 3, 24, 48, 168]
MA_WINDOWS = [6, 12, 24, 168]

def get_combined_data(client, city):
    """
    Bước 1: Hút dữ liệu cũ và mới, gộp lại thành 1 dòng thời gian liên tục
    """
    print(f"   📥 [Data] Đang tổng hợp dữ liệu cho {city}...")

    # 1.1 Load dữ liệu lịch sử (Giả sử file parquet hoặc csv lớn)
    # Nếu chưa có file lịch sử riêng, ta tạm thời bỏ qua hoặc load file csv gốc
    df_all = pd.DataFrame()

    # 1.2 Load dữ liệu Realtime từ MinIO (các file nhỏ cập nhật 3h/lần)
    # Lưu ý: Code này quét bucket raw để lấy dữ liệu mới nhất
    objects = client.list_objects(BUCKET_NEW, recursive=True)

    csv_list = []
    for obj in objects:
        # Lọc file csv, bỏ qua folder
        if not obj.object_name.endswith('.csv'): continue

        try:
            resp = client.get_object(BUCKET_NEW, obj.object_name)
            temp_df = pd.read_csv(io.BytesIO(resp.read()))
            resp.close()
            resp.release_conn()

            # Chuẩn hóa tên cột nếu cần (về lowercase)
            temp_df.columns = [c.lower() for c in temp_df.columns]

            # Lọc theo thành phố
            if 'city' in temp_df.columns:
                city_df = temp_df[temp_df['city'] == city]
                if not city_df.empty:
                    csv_list.append(city_df)
        except Exception as e:
            print(f"      Lỗi đọc file {obj.object_name}: {e}")

    if csv_list:
        df_new = pd.concat(csv_list)
        df_all = pd.concat([df_all, df_new])

    if df_all.empty:
        return None

    # Xử lý thời gian và sort
    # Giả sử cột thời gian tên là 'timestamp' hoặc 'time'
    time_col = 'timestamp' if 'timestamp' in df_all.columns else 'time' # Fallback
    df_all[time_col] = pd.to_datetime(df_all[time_col])
    df_all = df_all.sort_values(time_col).drop_duplicates(subset=[time_col], keep='last')

    # Đặt lại index là timestamp để dễ tính toán
    df_all = df_all.set_index(time_col)

    return df_all

def create_features(df):
    """
    Bước 2: Feature Engineering (Tạo đặc trưng để Model học)
    Phải khớp với logic lúc train lần đầu.
    """
    df = df.copy()

    # 2.1 Đặc trưng thời gian (Cyclic)
    df['hour'] = df.index.hour
    df['month'] = df.index.month
    df['day_of_week'] = df.index.dayofweek + 1

    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # 2.2 Lag Features (Quá khứ)
    target_col = 'temperature' # Giả sử cột mục tiêu là temperature hoặc temp_c
    if 'temp_c' in df.columns: target_col = 'temp_c' # Map về cột đúng

    for lag in LAG_PERIODS:
        df[f'temperature_lag_{lag}'] = df[target_col].shift(lag)

    # 2.3 Rolling Mean (Trung bình trượt)
    for window in MA_WINDOWS:
        df[f'temperature_ma_{window}h'] = df[target_col].rolling(window=window).mean()

    # Xóa các dòng NaN do shift tạo ra
    df = df.dropna()

    return df, target_col

def train_lightgbm(df, target_col):
    """
    Bước 3: Huấn luyện LightGBM
    """
    print(f"   ⚙️ [Train] Đang huấn luyện LightGBM...")

    # Xác định Features (loại bỏ cột target và các cột không dùng)
    exclude_cols = [target_col, 'city', 'weather_desc', 'text'] # Bỏ cột chữ
    features = [c for c in df.columns if c not in exclude_cols]

    X = df[features]
    y = df[target_col]

    # Khởi tạo model LightGBM
    model = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        n_jobs=-1
    )

    # Train trên TOÀN BỘ dữ liệu hiện có
    model.fit(X, y)

    return model, features

def forecast_recursive(model, last_data, features, days=7):
    """
    Bước 4: Dự báo tương lai (Recursive Forecast)
    Dùng kết quả dự đoán giờ T để làm đầu vào cho giờ T+1
    """
    print(f"   🔮 [Forecast] Đang dự báo {days} ngày tới...")

    future_preds = []
    current_step = last_data.copy() # Dòng dữ liệu cuối cùng của thực tế

    # Giả sử current_step là 1 Series, chuyển thành DataFrame 1 dòng
    if isinstance(current_step, pd.Series):
        current_step = current_step.to_frame().T

    last_time = current_step.index[0]

    # Loop từng giờ trong 7 ngày (7 * 24 = 168 giờ)
    for i in range(days * 24):
        next_time = last_time + timedelta(hours=i+1)

        # 1. Cập nhật Time Features cho giờ tiếp theo
        current_step.index = [next_time]
        current_step['hour'] = next_time.hour
        current_step['hour_sin'] = np.sin(2 * np.pi * next_time.hour / 24)
        current_step['hour_cos'] = np.cos(2 * np.pi * next_time.hour / 24)
        # (Cập nhật thêm month/dow nếu cần thiết...)

        # 2. Dự báo
        # Chỉ lấy đúng cột features mà model cần
        X_pred = current_step[features]
        pred_temp = model.predict(X_pred)[0]

        # 3. Lưu kết quả
        future_preds.append({
            "timestamp": next_time,
            "predicted_temp": round(pred_temp, 2)
        })

        # 4. CẬP NHẬT LAG & MA CHO VÒNG LẶP SAU (Quan trọng!)
        # Ở bước này, trong thực tế ta phải update lại các cột lag_1, lag_2...
        # bằng chính giá trị pred_temp vừa dự đoán được.
        # (Đây là mô phỏng đơn giản, trong production cần logic update rolling window phức tạp hơn)

        # Ví dụ update lag 1:
        if 'temperature_lag_1' in current_step.columns:
            current_step['temperature_lag_1'] = pred_temp

    return pd.DataFrame(future_preds)

def main():
    print("🚀 BẮT ĐẦU QUY TRÌNH: CẬP NHẬT DỮ LIỆU -> TRAIN LIGHTGBM -> DỰ BÁO")
    print("-" * 70)

    client = Minio(**MINIO_CONF)

    for city in CITIES:
        print(f"\n🏙️  THÀNH PHỐ: {city}")

        # 1. Lấy dữ liệu
        df_raw = get_combined_data(client, city)
        if df_raw is None or len(df_raw) < 168: # Cần ít nhất 7 ngày để tính MA_168h
            print("   ⚠️ Không đủ dữ liệu lịch sử để chạy feature engineering.")
            continue

        print(f"      -> Tổng dữ liệu: {len(df_raw)} dòng.")

        # 2. Chế biến Features
        df_processed, target_col = create_features(df_raw)

        # 3. Train LightGBM (Học lại từ đầu với dữ liệu mới nhất)
        model, feature_names = train_lightgbm(df_processed, target_col)

        # --- Lưu Model mới lên MinIO ---
        model_filename = f"LGBM_Production_{city.replace(' ', '_')}.pkl"
        bytes_file = io.BytesIO()
        joblib.dump(model, bytes_file)
        bytes_file.seek(0)
        client.put_object(BUCKET_MODEL, model_filename, bytes_file, length=bytes_file.getbuffer().nbytes)
        print(f"   💾 Đã lưu model mới: {model_filename}")

        # 4. Dự báo 7 ngày
        last_row = df_processed.iloc[[-1]] # Lấy dòng cuối cùng làm điểm tựa
        forecast_df = forecast_recursive(model, last_row, feature_names)

        # --- Lưu kết quả dự báo lên MinIO ---
        forecast_filename = f"forecast_7days_{city.replace(' ', '_')}.csv"
        csv_bytes = forecast_df.to_csv(index=False).encode('utf-8')
        client.put_object(
            BUCKET_OUTPUT,
            forecast_filename,
            io.BytesIO(csv_bytes),
            length=len(csv_bytes),
            content_type='application/csv'
        )

        # In thử kết quả ngày mai
        avg_tomorrow = forecast_df.iloc[:24]['predicted_temp'].mean()
        print(f"   ✅ Đã xong! Nhiệt độ trung bình ngày mai dự báo: {avg_tomorrow:.2f}°C")

    print("\n🎉 TOÀN BỘ QUY TRÌNH HOÀN TẤT!")

if __name__ == "__main__":
    main()

🚀 BẮT ĐẦU QUY TRÌNH: CẬP NHẬT DỮ LIỆU -> TRAIN LIGHTGBM -> DỰ BÁO
----------------------------------------------------------------------

🏙️  THÀNH PHỐ: Can Tho
   📥 [Data] Đang tổng hợp dữ liệu cho Can Tho...
   ⚠️ Không đủ dữ liệu lịch sử để chạy feature engineering.

🏙️  THÀNH PHỐ: Da Nang
   📥 [Data] Đang tổng hợp dữ liệu cho Da Nang...
   ⚠️ Không đủ dữ liệu lịch sử để chạy feature engineering.

🏙️  THÀNH PHỐ: Ha Noi
   📥 [Data] Đang tổng hợp dữ liệu cho Ha Noi...
   ⚠️ Không đủ dữ liệu lịch sử để chạy feature engineering.

🏙️  THÀNH PHỐ: Hai Phong
   📥 [Data] Đang tổng hợp dữ liệu cho Hai Phong...
   ⚠️ Không đủ dữ liệu lịch sử để chạy feature engineering.

🎉 TOÀN BỘ QUY TRÌNH HOÀN TẤT!
